# Sesión 3 — Programación Dinámica y Decisiones Logísticas

En este notebook se implementan los casos trabajados en la presentación. El objetivo es observar cómo una decisión secuencial puede representarse computacionalmente mediante **estados, decisiones, transiciones y valores acumulados**.

Trabajaremos tres aplicaciones:
1. Ruta de distribución mediante programación dinámica.
2. Carga de embarques mediante el modelo de mochila.
3. Planeación de producción e inventarios.

## 1. Librerías

Utilizaremos estructuras básicas de Python, `pandas` para mostrar resultados y `matplotlib`/`networkx` para representar la red.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Parte 1 — Ruta de distribución

Se enviará un contenedor desde el nodo **A** hasta el nodo **J**. La red está organizada por etapas y cada arco tiene asociado un costo de transporte.

Primero representaremos los costos de la red.

In [ ]:
costos = {
    "A": {"B": 2, "C": 2, "D": 5},
    "B": {"E": 7, "F": 4, "G": 6},
    "C": {"E": 3, "F": 2, "G": 4},
    "D": {"E": 4, "F": 1, "G": 5},
    "E": {"H": 4, "I": 8},
    "F": {"H": 6, "I": 3},
    "G": {"H": 3, "I": 4},
    "H": {"J": 5},
    "I": {"J": 3}
}

etapas = [
    ["A"],
    ["B", "C", "D"],
    ["E", "F", "G"],
    ["H", "I"],
    ["J"]
]

costos

## 2. Visualización de la red

Antes de resolver el problema, representaremos gráficamente las alternativas disponibles entre cada etapa.

In [ ]:
G = nx.DiGraph()

for origen, destinos in costos.items():
    for destino, costo in destinos.items():
        G.add_edge(origen, destino, costo=costo)

pos = {
    "A": (0, 1),
    "B": (2, 2), "C": (2, 1), "D": (2, 0),
    "E": (4, 2), "F": (4, 1), "G": (4, 0),
    "H": (6, 1.6), "I": (6, 0.4),
    "J": (8, 1)
}

plt.figure(figsize=(13, 6))
nx.draw_networkx_nodes(G, pos, node_size=1800)
nx.draw_networkx_labels(G, pos, font_size=11)
nx.draw_networkx_edges(
    G, pos, arrows=True, arrowsize=20,
    min_source_margin=20, min_target_margin=20
)

etiquetas = nx.get_edge_attributes(G, "costo")
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=etiquetas,
    font_size=9, rotate=False
)

plt.title("Red de distribución: costos por arco")
plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Solución mediante programación dinámica hacia atrás

Comenzaremos en el destino **J** y calcularemos el menor costo restante para cada nodo. Además del costo, guardaremos la mejor decisión para reconstruir posteriormente la ruta completa.

In [ ]:
valor = {"J": 0}
decision = {}

for etapa in reversed(etapas[:-1]):
    for nodo in etapa:
        alternativas = {}

        for siguiente, costo in costos[nodo].items():
            alternativas[siguiente] = costo + valor[siguiente]

        mejor_siguiente = min(alternativas, key=alternativas.get)

        valor[nodo] = alternativas[mejor_siguiente]
        decision[nodo] = mejor_siguiente

valor

Podemos mostrar los resultados de cada estado en una tabla. El valor representa el **costo mínimo desde ese nodo hasta el destino J**.

In [ ]:
tabla_valores = pd.DataFrame({
    "Nodo": list(valor.keys()),
    "Costo mínimo hasta J": list(valor.values())
}).sort_values("Costo mínimo hasta J")

tabla_valores

## 4. Reconstrucción de la ruta óptima

Una vez calculado el mejor siguiente nodo para cada estado, recorremos las decisiones desde **A** hasta llegar a **J**.

In [ ]:
ruta = ["A"]
actual = "A"

while actual != "J":
    actual = decision[actual]
    ruta.append(actual)

costo_total = valor["A"]

print("Ruta óptima:", " → ".join(ruta))
print(f"Costo mínimo total: ${costo_total:.2f} USD")

## 5. Desglose del costo de la ruta

Ahora verificaremos el costo acumulado arco por arco.

In [ ]:
detalle_ruta = []

for origen, destino in zip(ruta[:-1], ruta[1:]):
    costo = costos[origen][destino]
    detalle_ruta.append([origen, destino, costo])

df_ruta = pd.DataFrame(
    detalle_ruta,
    columns=["Origen", "Destino", "Costo"]
)

df_ruta.loc["Total", "Costo"] = df_ruta["Costo"].sum()
df_ruta

## 6. Visualización de la solución

Mostraremos nuevamente la red, destacando únicamente la ruta seleccionada por el algoritmo mediante un mayor grosor de línea.

In [ ]:
arcos_optimos = list(zip(ruta[:-1], ruta[1:]))

plt.figure(figsize=(13, 6))

nx.draw_networkx_nodes(G, pos, node_size=1800)
nx.draw_networkx_labels(G, pos, font_size=11)

otros_arcos = [e for e in G.edges() if e not in arcos_optimos]

nx.draw_networkx_edges(
    G, pos,
    edgelist=otros_arcos,
    width=1,
    alpha=0.25,
    arrows=True,
    arrowsize=15,
    min_source_margin=20,
    min_target_margin=20
)

nx.draw_networkx_edges(
    G, pos,
    edgelist=arcos_optimos,
    width=4,
    arrows=True,
    arrowsize=25,
    min_source_margin=20,
    min_target_margin=20
)

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=etiquetas,
    font_size=9,
    rotate=False
)

plt.title(f"Ruta óptima: {' → '.join(ruta)} | Costo total: ${costo_total:.2f}")
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Comparación con una ruta alternativa

La presentación utiliza **A → B → E → H → J** como ruta de referencia. Calcularemos su costo y el ahorro obtenido mediante la solución óptima.

In [ ]:
ruta_referencia = ["A", "B", "E", "H", "J"]

costo_referencia = sum(
    costos[o][d]
    for o, d in zip(ruta_referencia[:-1], ruta_referencia[1:])
)

ahorro = costo_referencia - costo_total
porcentaje_ahorro = ahorro / costo_referencia * 100

print(f"Ruta de referencia: {' → '.join(ruta_referencia)}")
print(f"Costo de referencia: ${costo_referencia:.2f}")
print(f"Costo óptimo: ${costo_total:.2f}")
print(f"Ahorro: ${ahorro:.2f}")
print(f"Reducción porcentual: {porcentaje_ahorro:.2f}%")

# Parte 2 — Carga de embarques: modelo de mochila

El vehículo tiene una capacidad máxima de **10 toneladas**. Para cada lote debemos decidir si se carga o no, buscando maximizar el valor económico transportado.

In [ ]:
lotes = ["Lote 1", "Lote 2", "Lote 3", "Lote 4"]
pesos = [3, 4, 5, 6]
valores = [400, 500, 700, 800]
capacidad = 10

mercancias = pd.DataFrame({
    "Lote": lotes,
    "Peso (t)": pesos,
    "Valor ($)": valores
})

mercancias["Valor por tonelada"] = (
    mercancias["Valor ($)"] / mercancias["Peso (t)"]
)

mercancias

## 8. Tabla de programación dinámica

Construiremos una tabla donde las filas representan los lotes evaluados y las columnas la capacidad disponible del vehículo.

In [ ]:
n = len(lotes)

dp = np.zeros((n + 1, capacidad + 1), dtype=int)

for i in range(1, n + 1):
    peso = pesos[i - 1]
    valor_lote = valores[i - 1]

    for w in range(capacidad + 1):
        if peso <= w:
            dp[i, w] = max(
                dp[i - 1, w],
                valor_lote + dp[i - 1, w - peso]
            )
        else:
            dp[i, w] = dp[i - 1, w]

tabla_dp = pd.DataFrame(
    dp,
    index=["Sin lotes"] + lotes,
    columns=[f"{w} t" for w in range(capacidad + 1)]
)

tabla_dp

## 9. Reconstrucción de los lotes seleccionados

La última celda contiene el mejor valor posible. Retrocederemos en la tabla para identificar qué lotes generan esa solución.

In [ ]:
seleccionados = []
w = capacidad

for i in range(n, 0, -1):
    if dp[i, w] != dp[i - 1, w]:
        seleccionados.append(i - 1)
        w -= pesos[i - 1]

seleccionados.reverse()

peso_total = sum(pesos[i] for i in seleccionados)
valor_total = sum(valores[i] for i in seleccionados)

print("Lotes seleccionados:")
for i in seleccionados:
    print(f"- {lotes[i]}: {pesos[i]} t | ${valores[i]}")

print(f"\nPeso total: {peso_total} t")
print(f"Ocupación: {peso_total / capacidad * 100:.1f}%")
print(f"Valor máximo: ${valor_total}")

### Observación

El cálculo computacional permite validar directamente todas las combinaciones implícitas en la recurrencia. Compare el resultado obtenido con el desarrollado manualmente en la presentación e identifique qué decisión explica cualquier diferencia.

# Parte 3 — Planeación de producción e inventarios

Utilizaremos los parámetros del caso de cuatro meses. El estado será el inventario disponible y la decisión será cuántas unidades producir en cada periodo.

Para respetar el planteamiento, el inventario final de cada mes no podrá superar **20 unidades**.

In [ ]:
demanda = [20, 30, 40, 20]

K = 100   # costo fijo de arranque
c = 10    # costo variable por unidad
h = 2     # costo de inventario por unidad
S = 20    # inventario máximo

print("Demanda total:", sum(demanda))

## 10. Función de costo y estados factibles

Resolveremos el problema hacia atrás. Para cada mes e inventario inicial, evaluaremos las cantidades de producción que satisfacen la demanda y mantienen el inventario dentro del límite.

In [ ]:
from functools import lru_cache

@lru_cache(None)
def resolver_periodo(mes, inventario_inicial):
    if mes == len(demanda):
        return (0, []) if inventario_inicial == 0 else (float("inf"), [])

    mejor_costo = float("inf")
    mejor_plan = []

    d = demanda[mes]

    # El inventario final debe quedar entre 0 y S.
    # Por ello basta evaluar la producción que conduce a esos estados.
    for inventario_final in range(S + 1):
        produccion = d + inventario_final - inventario_inicial

        if produccion < 0:
            continue

        costo_arranque = K if produccion > 0 else 0
        costo_periodo = (
            costo_arranque
            + c * produccion
            + h * inventario_final
        )

        costo_futuro, plan_futuro = resolver_periodo(
            mes + 1,
            inventario_final
        )

        costo_total = costo_periodo + costo_futuro

        if costo_total < mejor_costo:
            mejor_costo = costo_total
            mejor_plan = [{
                "Mes": mes + 1,
                "Inventario inicial": inventario_inicial,
                "Demanda": d,
                "Producción": produccion,
                "Inventario final": inventario_final,
                "Costo del periodo": costo_periodo
            }] + plan_futuro

    return mejor_costo, mejor_plan

costo_optimo, plan_optimo = resolver_periodo(0, 0)

print(f"Costo total mínimo: ${costo_optimo:.2f}")

## 11. Plan óptimo de producción

Mostraremos las decisiones obtenidas periodo por periodo para revisar el balance entre producción, demanda e inventario.

In [ ]:
df_plan = pd.DataFrame(plan_optimo)
df_plan

## 12. Verificación del límite de inventario

Además del costo total, es importante comprobar que la solución respete las restricciones operativas del modelo.

In [ ]:
print("Inventario máximo observado:", df_plan["Inventario final"].max())
print("Límite permitido:", S)
print(
    "¿La solución respeta el límite?",
    bool((df_plan["Inventario final"] <= S).all())
)

print(
    "¿El inventario final del horizonte es cero?",
    df_plan.iloc[-1]["Inventario final"] == 0
)

## 13. Visualización del plan

La gráfica permite observar cómo se coordinan la producción, la demanda y el inventario a lo largo de los cuatro meses.

In [ ]:
meses = df_plan["Mes"].to_numpy()

plt.figure(figsize=(10, 5))
plt.plot(meses, df_plan["Demanda"], marker="o", label="Demanda")
plt.plot(meses, df_plan["Producción"], marker="o", label="Producción")
plt.plot(meses, df_plan["Inventario final"], marker="o", label="Inventario final")

plt.xticks(meses)
plt.xlabel("Mes")
plt.ylabel("Unidades")
plt.title("Plan óptimo de producción e inventario")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Parte 4 — Ejercicio para el alumno

Una empresa debe seleccionar mercancías para un vehículo con capacidad máxima de **15 toneladas**.

| Lote | Peso (t) | Valor ($) |
|---|---:|---:|
| A | 2 | 300 |
| B | 5 | 700 |
| C | 6 | 900 |
| D | 4 | 600 |
| E | 7 | 1,000 |

### Actividad

1. Identifique las **etapas**, el **estado** y la **decisión**.
2. Construya la tabla de programación dinámica.
3. Determine los lotes que deben cargarse.
4. Calcule el peso utilizado y el valor máximo transportado.
5. Explique por qué seleccionar únicamente los lotes con mayor valor individual no necesariamente genera la mejor solución.

# Cierre

Los tres casos utilizan la misma lógica general de programación dinámica: **estado actual → decisión → transición → valor futuro**. Lo que cambia entre aplicaciones es la definición del estado, las decisiones factibles y la función que se desea optimizar.